## Environment setup

This study uses CLaP as an external detector. Its dependencies are deliberately
optional rather than part of FeatureGraph's core installation. From the
repository root, run:

```python
%pip install -e ".[clap-study]"
```

Then restart the notebook kernel before running the study. The equivalent
requirements-file installation is `%pip install -r notebooks/clap_requirements.txt`.

In [ ]:
# CLaP state-occurrence study: complete human-authored scientific input
#
# This notebook is the authoritative scientific specification. CLaP supplies
# inferred state labels. FeatureGraph preserves those labels as bounded object
# occurrences; it does not claim to detect or interpret the latent states.

import featuregraph as fg
try:
    from claspy.data_loader import load_tssb_dataset
    from claspy.state_detection import AgglomerativeCLaPDetection
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "The CLaP study dependencies are not installed in this kernel. "
        "Run `%pip install -e '.[clap-study]'` from the repository root, "
        "restart the kernel, and run the notebook again."
    ) from exc
import pandas as pd

study_scope = {
    "paper": "CLaP - State Detection from Time Series (Ermshaus, Schäfer, Leser)",
    "detector": "claspy.state_detection.AgglomerativeCLaPDetection",
    "detector_version": "0.2.8",
    "dataset": "Crop from the Time Series Segmentation Benchmark",
    "dataset_loader": "claspy.data_loader.load_tssb_dataset",
    "purpose": (
        "Preserve one externally inferred CLaP state sequence as explicit, "
        "bounded FeatureGraph state-occurrence objects."
    ),
    "unit_of_analysis": "one Crop sensor time series",
}

detector_contract = {
    "configuration": "AgglomerativeCLaPDetection defaults; no FeatureGraph tuning",
    "materializer": "featuregraph.from_state_sequence",
    "input": "the one-dimensional Crop time series returned by load_tssb_dataset",
    "output": "one inferred integer state label per observation",
    "authority_boundary": (
        "CLaP determines inferred state membership; FeatureGraph must not change, "
        "smooth, merge, relabel, or reinterpret the returned state sequence."
    ),
}

# CLaP remains authoritative for label membership. This contract only
# compiles that external categorical sequence into deterministic occurrence
# identity; no label expression, smoothing, or interpretation is introduced.
state_contract = {
    "version": "state-contract-v1",
    "state_column": "state_label",
    "events": {},
    "boundary_policy": {
        "include_first_entry": True,
        "include_last_exit": True,
    },
}

observation_definition = {
    "sample_index": "zero-based position in the Crop time series",
    "signal_raw": "unaltered Crop observation value",
    "clap_state": "CLaP inferred state label at the observation",
    "reference_state": "benchmark reference state label at the observation",
}

event_definitions = {
    "enter_clap_state": (
        "true at sample 0 and whenever clap_state differs from the preceding sample"
    ),
    "exit_clap_state": (
        "true at the final sample and whenever clap_state differs from the following sample"
    ),
    "clap_change_point": (
        "sample index of every enter_clap_state event except the first observation"
    ),
}

object_definition = {
    "name": "clap_state_occurrence",
    "identity": "cumulative count of enter_clap_state events, zero based",
    "membership": "one maximal contiguous run with a constant CLaP state label",
    "interval": "half-open [start_index, end_index_exclusive)",
    "complete": (
        "an internal run bounded by detected state changes on both sides"
    ),
    "boundary_fragment": (
        "the first or final run, whose outer boundary is the observation-series edge"
    ),
    "class_instance_distinction": (
        "clap_state is a recurring class; occurrence_id identifies one bounded instance"
    ),
}

object_properties = {
    "occurrence_id": "zero-based within-series identity",
    "clap_state": "constant inferred class label for the occurrence",
    "start_index": "first included sample",
    "end_index": "last included sample",
    "end_index_exclusive": "first excluded sample",
    "duration_samples": "end_index_exclusive - start_index",
    "signal_minimum": "minimum raw signal in the occurrence",
    "signal_maximum": "maximum raw signal in the occurrence",
    "signal_mean": "mean raw signal in the occurrence",
    "signal_std": "population standard deviation of the raw signal",
    "previous_state": "CLaP class of the preceding occurrence, when present",
    "next_state": "CLaP class of the following occurrence, when present",
    "boundary_fragment": "true for first and final occurrences",
}

relation_definition = {
    "name": "precedes",
    "source": "each occurrence except the final occurrence",
    "target": "the immediately following occurrence",
    "properties": ["source_state", "target_state", "boundary_index"],
}

validation_requirements = [
    "raw observations and CLaP labels retain their original order and values",
    "exactly one state label is present for every observation",
    "each occurrence contains one constant CLaP state and contiguous samples",
    "occurrences partition every source observation exactly once",
    "reconstructing labels from occurrences exactly reproduces the CLaP sequence",
    "entry events after sample zero equal the CLaP change-point indices",
    "relations connect every adjacent pair of occurrences exactly once",
    "the relation state pairs equal the sparse CLaP transition graph",
    "first and final occurrences remain explicit boundary fragments",
]

requested_outputs = [
    "observation table with raw signal, reference state, CLaP state, events, and identity",
    "one-row-per-occurrence object table",
    "adjacent-occurrence relation table",
    "CLaP versus reference boundary comparison",
    "permutation-invariant state agreement metrics",
    "validation report and provenance",
]

supported_claims = [
    "featuregraph.from_state_sequence preserves a CLaP state sequence as explicit bounded occurrences",
    "every occurrence can be traced to supporting observations and CLaP provenance",
    "recurring state classes remain distinct from their individual temporal occurrences",
    "boundary and state agreement with the benchmark can be measured without relabeling CLaP output",
]

unsupported_claims = [
    "FeatureGraph discovered or improved the latent states",
    "the inferred state numbers have inherent semantic meaning",
    "the Crop example establishes general interoperability with CLaP",
    "FeatureGraph is a competitor or replacement for CLaP",
    "the benchmark reference labels are infallible physical ground truth",
]

execution_contract = {
    "allowed": [
        "run the documented CLaP Crop example",
        "materialize the declared events, objects, relations, measurements, and checks",
        "calculate boundary error and permutation-invariant agreement metrics",
        "record package versions and output counts",
    ],
    "must_ask_before": [
        "changing CLaP parameters or dataset selection",
        "smoothing, merging, splitting, or relabeling the inferred sequence",
        "discarding boundary fragments",
        "assigning crop semantics to inferred integer labels",
        "extending claims beyond supported_claims",
    ],
}
